app password: https://myaccount.google.com/apppasswords


In [10]:
from dotenv import load_dotenv
import requests
from agents import Agent, Runner, trace, function_tool, ModelSettings,OpenAIChatCompletionsModel
from agents.extensions.visualization import draw_graph
from openai.types.responses import ResponseTextDeltaEvent
import os
import asyncio
import smtplib
from email.message import EmailMessage
load_dotenv(override=True)
from openai import AsyncOpenAI

In [11]:
anthropic_client = AsyncOpenAI(
    api_key=os.environ["ANTHROPIC_API_KEY"],
    base_url="https://api.anthropic.com/v1/",
)
   
anthropic_model = OpenAIChatCompletionsModel(
        model="claude-haiku-4-5-20251001",
        openai_client=anthropic_client,
    )

In [12]:
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")
EMAIL_ADDRESS_TO = os.getenv("EMAIL_ADDRESS_TO")

if EMAIL_ADDRESS:
    print("Email address is set")
else:
    print("Email address is not set")

if EMAIL_SMTP_SERVER:
    print("SMTP server is set")
else:
    print("SMTP server is not set")

if EMAIL_APP_PASSWORD:
    print("App password is set")
else:
    print("App password is not set")

USE_EMAIL = EMAIL_ADDRESS and EMAIL_SMTP_SERVER and EMAIL_APP_PASSWORD

if USE_EMAIL:
    print("Email is set up and we will try using it")
else:
    print("Email is not set up; we will send push notifications instead")

Email address is set
SMTP server is set
App password is set
Email is set up and we will try using it


In [13]:
def send_email(subject, text_body, html_body):
    msg = EmailMessage()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = EMAIL_ADDRESS_TO
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)

In [14]:
send_email("Testing testing 123", "Fingers crossed..", "<html><body><strong>Fingers</strong> crossed..</body></html>")

In [15]:
def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)

In [16]:
send_message("Big news", "Communications are a go!", "<html><body>Communications are a <strong>go!</strong></body></html>")

## Agent Orchestration

There are 2 models for Agent Orchestration; by code and by LLMs.

By code: more predictable and deterministic.

By LLMs: more powerful.

https://openai.github.io/openai-agents-python/multi_agent/


## Orchestrating by Code


In [17]:
intro = """
You are a sales agent working for ComplAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write emails.
"""

instructions1 = intro + "Your email style is professional, serious, with gravitas and credibility."
instructions2 = intro + "Your email style is witty, engaging, and humorous."
instructions3 = intro + "Your email style is concise, to the point, in the style of a busy senior executive."

In [18]:
sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model=anthropic_model)
sales_agent2 = Agent(name="Humorous Sales Agent", instructions=instructions2, model=anthropic_model)
sales_agent3 = Agent(name="Executive Sales Agent", instructions=instructions3, model=anthropic_model)


In [19]:

result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

**Subject: Reducing Your SOC2 Audit Timeline from Months to Weeks**

Dear [Prospect Name],

I work with SaaS companies that are preparing for SOC2 audits—and I've noticed a pattern: most teams spend 3-6 months scrambling to organize evidence, document controls, and address auditor findings.

It doesn't have to be this way.

ComplAI uses AI to automate the compliance workload. Instead of your team manually compiling documentation and chasing down control evidence, our platform continuously monitors your infrastructure, identifies gaps, and generates audit-ready reports.

The result: companies typically move from "compliance chaos" to "audit-ready" in 4-8 weeks, not months.

I work specifically with founders and compliance leaders at Series B-C companies processing customer data or operating under enterprise contracts. If you're either preparing for your first SOC2 audit or managing the ongoing compliance burden, I think we could meaningfully reduce your timeline and team strain.

Would 

In [20]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")


**Subject: Reducing Your SOC2 Audit Risk**

Dear [Prospect Name],

Most organizations underestimate the operational and financial burden of SOC2 compliance until they're facing an audit deadline.

The traditional approach—spreadsheets, manual documentation, and fragmented processes—leaves critical gaps. Auditors find them. Remediation becomes costly.

At ComplAI, we've built a different solution. Our AI-powered platform automates evidence collection, identifies compliance gaps before auditors do, and consolidates your entire audit-readiness posture into a single source of truth. Organizations using ComplAI reduce audit preparation time by 60% while strengthening their control environment.

This matters because:
- **Audit failure costs time and reputation**—both expensive
- **Manual compliance processes don't scale** as you grow
- **AI catches what spreadsheets miss**—systematically

I've worked with teams across [your industry] who discovered significant gaps only weeks before their au

In [21]:
decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Do not give an explanation; reply with the selected email only.
"""

sales_picker = Agent(name="Sales_picker", instructions=decision, model=anthropic_model)


In [22]:
message = "Write a cold sales email"

with trace("Sales selection workflow"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")


Best sales email:
**Subject: SOC2 Compliance Without the Headaches**

Dear [Prospect Name],

I hope this message finds you well.

Most SaaS companies we speak with describe SOC2 compliance preparation the same way: time-consuming, resource-intensive, and fraught with uncertainty about what auditors will actually require.

ComplAI changes that equation.

We've built an AI-powered platform that transforms SOC2 compliance from a reactive scramble into a managed, predictable process. Our tool automatically maps your security controls, identifies gaps before auditors do, and generates the documentation you'll actually need—cutting typical preparation time from months to weeks.

For growing SaaS companies managing multiple compliance initiatives, this typically translates to:
- 60-70% reduction in internal audit preparation time
- Significantly higher first-pass audit success rates
- Clear visibility into compliance status at any moment

I won't claim this works for every organization—it's b

In [23]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_message(subject, text_body, html_body)
    return "Email sent successfully"

In [24]:
send_email_tool.params_json_schema

{'properties': {'subject': {'description': 'The subject of the email',
   'title': 'Subject',
   'type': 'string'},
  'text_body': {'description': 'The body of the email as plain text',
   'title': 'Text Body',
   'type': 'string'},
  'html_body': {'description': 'The HTML body of the email',
   'title': 'Html Body',
   'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [25]:
decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Then use your tool to send the email.
"""

require_tool = ModelSettings(tool_choice="required")

sales_sender = Agent(name="Sales Sender", instructions=decision, model=anthropic_model, tools=[send_email_tool], model_settings=require_tool)

In [26]:
message = "Write a cold sales email"

with trace("Sales selection workflow with sending"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    response = await Runner.run(sales_sender, emails)

    print(f"Final response:\n{response.final_output}")

Final response:
I chose the **third email (90-day SOC2 readiness check)** as the best cold sales email.

**Here's why** (from a customer perspective):

1. **Respects my time** – It's concise and doesn't waste words. The main pitch is under 80 words.

2. **Opens with a problem, not a pitch** – It starts with a question about *my* pain ("How much time is your team spending?") rather than immediately selling something.

3. **Uses credible specifics** – "60% compliance overhead reduction" and "pass audits on the first attempt" are concrete metrics that feel earned, not made-up promises.

4. **Gives me an easy out** – Offers a 2-minute video as an alternative to a call. This removes friction and shows respect for my schedule.

5. **Clear, small ask** – 15 minutes is reasonable and non-threatening.

6. **Honest tone** – No emojis, puns, or false friendliness. Just straightforward value.

The first email is well-written but too long and formal. The second email uses humor and personality ("gl

## Part 3: Orchestrating by LLMs

### 3a: via Tools

The simplest way to have 1 Agent choose to invoke another is by treating it as a tool call.

Agent A -> Agent B -> Agent A


In [27]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_email_writer_1", tool_description=description)
tool1

FunctionTool(name='sales_email_writer_1', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_email_writer_1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001A96B841C60>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [28]:
tool1 = sales_agent1.as_tool(tool_name="sales_email_writer_1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_email_writer_2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_email_writer_3", tool_description=description)

tools = [tool1, tool2, tool3, send_email_tool]

tools

[FunctionTool(name='sales_email_writer_1', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_email_writer_1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001A96B8A5C60>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_email_writer_2', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_email_writer_2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_t

## Sales Manager - our planning agent


In [29]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_writer tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=anthropic_model)


In [30]:
with trace("Sales manager"):
    result = await Runner.run(sales_manager, task)